# Demo: pychameleon na zbiorze Aggregation

Krótkie demo działania pakietu `pychameleon` — kompletny pipeline w 5 krokach:

1. Wczytanie zbioru Aggregation (788 punktów, 7 klastrów, Gionis et al. 2007)
2. Wizualizacja danych wejściowych
3. Klasteryzacja jednym wywołaniem `Chameleon().fit_predict()`
4. Wizualizacja wyniku
5. Ocena ilościowa wzgl. ground truth (ARI, NMI)

## 1. Wczytanie danych

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from pychameleon import Chameleon
from pychameleon._datasets import load

plt.rcParams['figure.dpi'] = 100

ds = load('aggregation')
X, y_true = ds.X, ds.y

print(f'Zbiór:     {ds.name}')
print(f'Punktów:   {X.shape[0]}')
print(f'Wymiarów:  {X.shape[1]}')
print(f'Klastrów:  {len(np.unique(y_true))}')

## 2. Wizualizacja danych wejściowych

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[:, 0], X[:, 1], c='steelblue', s=15, linewidths=0)
ax.set_title('Aggregation — 788 punktów (bez etykiet)')
ax.set_xticks([]); ax.set_yticks([])
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 3. Klasteryzacja

Parametry pochodzą z grid-search HPO (`scripts/run_hpo.py`) — patrz dokumentacja, rozdz. 6.3.

In [ ]:
import time

model = Chameleon(
    n_clusters=7,
    k_nn=10,
    min_cluster_size=40,
    alpha=0.5,
)

t0 = time.perf_counter()
labels = model.fit_predict(X)
elapsed = time.perf_counter() - t0

print(f'Runtime:            {elapsed:.3f} s')
print(f'Znaleziono klastrów: {model.n_clusters_}')
print(f'Etykiety:            {np.unique(labels)}')

## 4. Wizualizacja wyniku — ground truth vs predykcja

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap='tab10', s=15, linewidths=0)
axes[0].set_title(f'Ground truth ({len(np.unique(y_true))} klastrów)')
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_aspect('equal')

axes[1].scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=15, linewidths=0)
axes[1].set_title(f'pychameleon ({model.n_clusters_} klastrów)')
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

## 5. Ocena ilościowa

- **ARI** (Adjusted Rand Index) — pairwise agreement vs GT, chance-corrected.
- **NMI** (Normalized Mutual Information) — wzajemna informacja, normalizowana.

Obie metryki są w zakresie `[0, 1]`; wartości bliskie 1 oznaczają wysoką zgodność z ground truth.

In [ ]:
ari = adjusted_rand_score(y_true, labels)
nmi = normalized_mutual_info_score(y_true, labels)
quality = (ari + nmi) / 2

print(f'ARI:     {ari:.3f}')
print(f'NMI:     {nmi:.3f}')
print(f'Quality: {quality:.3f}')

**Wynik:** algorytm poprawnie odtwarza wszystkie 7 klastrów z bardzo wysoką zgodnością z ground truth. Pełna analiza ilościowa na 7 datasetach w dokumentacji końcowej (rozdz. 6).